# NEU-DET 数据探索

本 notebook 用于探索 NEU-DET 数据集：
- 查看缺陷类别分布
- 可视化图像与标注框
- 预览 OpenCV 预处理效果
- 分析训练/验证/测试集划分

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from visionguard.utils.dataset_utils import ID_TO_CLASS

DATA_DIR = Path("../data/processed")
SPLIT = "train"

In [ ]:
# Count class distribution
from collections import Counter

label_dir = DATA_DIR / SPLIT / "labels"
class_counts = Counter()
for label_path in sorted(label_dir.glob("*.txt")):
    with open(label_path) as f:
        for line in f:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

print({ID_TO_CLASS[i]: class_counts[i] for i in sorted(class_counts)})

In [ ]:
# Visualize a sample image with annotations
def plot_image_with_boxes(image_path, label_path):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x, y, bw, bh = map(float, parts[1:])
            x1 = int((x - bw / 2) * w)
            y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w)
            y2 = int((y + bh / 2) * h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image, ID_TO_CLASS[class_id], (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

# Example usage (when data exists):
# img_path = next((DATA_DIR / SPLIT / "images").glob("*.jpg"))
# plot_image_with_boxes(img_path, DATA_DIR / SPLIT / "labels" / img_path.with_suffix(".txt").name)